In [1]:
! pip install -q qiskit qiskit_aer

In [2]:
import subprocess

def kill_processes_on_port(port):
    print(f"Attempting to kill processes on port {port}...")
    try:
        # Find PIDs using the specified port
        lsof_output = subprocess.check_output(['lsof', '-i', f':{port}'], stderr=subprocess.STDOUT, text=True)
        pids = []
        for line in lsof_output.splitlines():
            parts = line.split()
            # Ensure it's a line with process info and the PID is a digit
            if len(parts) > 1 and parts[1].isdigit():
                pids.append(parts[1])

        if pids:
            print(f"Killing processes on port {port}: {', '.join(pids)}")
            for pid in pids:
                subprocess.run(['kill', '-9', pid])
            print(f"Processes on port {port} killed successfully.")
        else:
            print(f"No processes found on port {port}.")
    except subprocess.CalledProcessError as e:
        # lsof returns exit code 1 if no entries are found, but also for errors
        if e.returncode == 1 and not e.output.strip():
            print(f"No processes found on port {port}.")
        elif e.output and ("No such file or directory" in e.output or "command not found" in e.output):
            print("lsof command not found. Please ensure it is installed or use an alternative method.")
        elif e.output and ("COMMAND" in e.output and "PID" in e.output) and e.returncode == 1:
            print(f"No processes found on port {port} (lsof returned header only or no entries).")
        else:
            print(f"Error checking/killing processes on port {port}: {e.output}")
    except Exception as e:
        print(f"An unexpected error occurred while processing port {port}: {e}")

# Kill processes on port 65434 (Eve)
kill_processes_on_port(65434)

# Kill processes on port 65435 (Bob)
kill_processes_on_port(65435)

Attempting to kill processes on port 65434...
Killing processes on port 65434: 38665, 38665, 38691
Processes on port 65434 killed successfully.
Attempting to kill processes on port 65435...
Killing processes on port 65435: 38614
Processes on port 65435 killed successfully.


In [3]:
with open('bob_server.py', 'w') as f:
    f.write('''
# Bob - run 1st

import socket
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer

backend = Aer.get_backend('aer_simulator')

def generate_key(size):
    return [random.randint(0, 1) for _ in range(size)]

def measure_circuit(circuit, basis):
    for i in range(len(basis)):
        if basis[i] == 1:
            circuit.h(i)
    circuit.measure(range(len(basis)), range(len(basis)))
    t_circuit = transpile(circuit, backend)
    job = backend.run(t_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(circuit)
    outcome = max(counts, key=counts.get)
    measurement = [int(outcome[i]) for i in range(len(basis))]
    return measurement

def main():
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.bind(('localhost', 65435))  # Port for Bob
    server_socket.listen(1)
    print("Bob is listening for connections...")

    try:
        while True:
            conn, addr = server_socket.accept()
            print(f"Connected to Eve from {addr}")

            try:
                while True:
                    # Receive data from Eve
                    data = conn.recv(1024).decode()
                    if not data:
                        raise ValueError("No data received from Eve.")

                    alice_measurement, alice_basis = data.split('|')
                    alice_measurement = list(map(int, alice_measurement.strip('[]').split(',')))
                    alice_basis = list(map(int, alice_basis.strip('[]').split(',')))

                    print(f"Bob received Alice's data: {alice_measurement}, {alice_basis}")

                    # Simulate Bob's measurement
                    bob_basis = generate_key(len(alice_measurement))
                    bob_circuit = QuantumCircuit(len(alice_measurement), len(alice_measurement))
                    for i, bit in enumerate(alice_measurement):
                        if bit == 1:
                            bob_circuit.x(i)
                        if bob_basis[i] == 1:
                            bob_circuit.h(i)

                    bob_measurement = measure_circuit(bob_circuit, bob_basis)
                    print(f"Bob's measurement: {bob_measurement}")

                    # Send Bob's measurement to Eve
                    message = str(bob_measurement)
                    conn.sendall(message.encode())
                    print(f"Message sent to Eve: {message}")

            except ValueError as ve:
                print(f"Value error: {ve}")

            finally:
                conn.close()

    except Exception as e:
        print(f"Server error: {e}")

    finally:
        server_socket.close()

if __name__ == "__main__":
    main()
''')
print("Bob's script saved to bob_server.py")

Bob's script saved to bob_server.py


In [4]:
with open('eve_server.py', 'w') as f:
    f.write('''
# Eve - run 2nd

import socket
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer

# Define the backend for the simulation
backend = Aer.get_backend('aer_simulator')

# Define Security Variables
eve = 0
total = 0

def generate_key(size):
    return [random.randint(0, 1) for _ in range(size)]

def encode_key(key):
    circuit = QuantumCircuit(len(key), len(key))
    for i, bit in enumerate(key):
        if bit == 1:
            circuit.x(i)
    return circuit

def measure_circuit(circuit, basis):
    for i in range(len(basis)):
        if basis[i] == 1:
            circuit.h(i)
    circuit.measure(range(len(basis)), range(len(basis)))
    t_circuit = transpile(circuit, backend)
    job = backend.run(t_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(circuit)
    outcome = max(counts, key=counts.get)
    measurement = [int(outcome[i]) for i in range(len(basis))]
    return measurement

def qkd_protocol(alice_key, bob_basis):
    alice_circuit = encode_key(alice_key)
    bob_circuit = alice_circuit.copy()

    bob_measurement = measure_circuit(bob_circuit, bob_basis)

    # Bob publicly announces his basis
    public_basis = bob_basis

    # Alice and Bob discard the bits where their bases don't match
    shared_key = []
    for i in range(len(alice_key)):
        if public_basis[i] == bob_basis[i]:
            shared_key.append(bob_measurement[i])

    return shared_key

def eve_intercept(alice_key, eve_basis):
    eve_measurement = measure_circuit(encode_key(alice_key), eve_basis)

    # Eve guesses the original key
    eve_guess = []
    for i in range(len(alice_key)):
        if eve_basis[i] == 0:
            eve_guess.append(eve_measurement[i])
        else:
            eve_guess.append(1 - eve_measurement[i])
    return eve_guess

def calculate_success_rate(total_attempts, successful_attempts):
    if total_attempts == 0:
        return 0
    return (successful_attempts / total_attempts) * 100

def main():
    global eve, total

    # Setup server socket to receive connections
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1) # Added to allow port reuse
    server_socket.bind(('localhost', 65434))  # Port for Eve
    server_socket.listen(1)
    print("Eve is listening for connections...")

    try:
        # Accept Alice's connection
        alice_conn, alice_addr = server_socket.accept()
        print(f"Connected to Alice from {alice_addr}")

        # Connect to Bob
        bob_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        bob_socket.connect(('localhost', 65435))  # Port for Bob
        print("Connected to Bob.")

        try:
            while True:
                # Receive data from Alice
                alice_data = alice_conn.recv(1024).decode()
                if not alice_data:
                    raise ValueError("No data received from Alice.")

                print(f"Eve received Alice's data: {alice_data}")

                # Forward Alice's data to Bob
                bob_socket.sendall(alice_data.encode())
                print(f"Data sent to Bob: {alice_data}")

                # Receive Bob's response
                bob_response = bob_socket.recv(1024).decode()
                if not bob_response:
                    raise ValueError("No response received from Bob.")

                print(f"Eve received Bob's response: {bob_response}")

                # Send Bob's response back to Alice
                alice_conn.sendall(bob_response.encode())
                print(f"Response sent to Alice: {bob_response}")

                # Track success rate
                alice_key, _ = list(map(int, alice_data.split('|')[0].strip('[]').split(','))), len(alice_data.split('|')[1].strip('[]').split(','))
                bob_measurement = list(map(int, bob_response.strip('[]').split(',')))
                shared_key = qkd_protocol(alice_key, bob_measurement)

                eve_guess = eve_intercept(alice_key, bob_measurement)

                if shared_key == eve_guess:
                    eve += 1

                total += 1
                success_rate = calculate_success_rate(total, eve)
                print(f"Iteration: {total}, Eve's success rate: {success_rate:.5f}%")

        except ValueError as ve:
            print(f"Value error: {ve}")

        finally:
            alice_conn.close()
            bob_socket.close()

    except Exception as e:
        print(f"Server error: {e}")

    finally:
        server_socket.close()

if __name__ == "__main__":
    main()
''')
print("Eve's script saved to eve_server.py")

Eve's script saved to eve_server.py


In [5]:
with open('alice_client.py', 'w') as f:
    f.write('''
# Alice - run 3rd

import socket
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from time import sleep

backend = Aer.get_backend('aer_simulator')

def generate_key(size):
    return [random.randint(0, 1) for _ in range(size)]

def measure_circuit(circuit, basis):
    for i in range(len(basis)):
        if basis[i] == 1:
            circuit.h(i)
    circuit.measure(range(len(basis)), range(len(basis)))
    t_circuit = transpile(circuit, backend)
    job = backend.run(t_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(circuit)
    outcome = max(counts, key=counts.get)
    measurement = [int(outcome[i]) for i in range(len(basis))]
    return measurement

def main():
    client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client_socket.connect(('localhost', 65434))  # Connect to Eve
    print("Connected to Eve.")

    try:
        while True:
            # Generate Alice's random key and basis
            alice_key = generate_key(8)  # Example size of 8
            alice_basis = generate_key(len(alice_key))
            alice_circuit = QuantumCircuit(len(alice_key), len(alice_key))

            # Prepare Alice's state
            for i, bit in enumerate(alice_key):
                if bit == 1:
                    alice_circuit.x(i)
                if alice_basis[i] == 1:
                    alice_circuit.h(i)

            alice_measurement = measure_circuit(alice_circuit, alice_basis)
            print(f"Alice's measurement: {alice_measurement}")

            # Send Alice's data to Eve
            message = f"{alice_measurement}|{alice_basis}"
            client_socket.sendall(message.encode())
            print("Alice's data sent to Eve.")

            # Receive the shared key from Eve
            try:
                eve_data = client_socket.recv(1024).decode()
                if not eve_data:
                    raise ValueError("No data received from Eve.")
                print(f"Received shared key from Eve: {eve_data}")

            except ValueError as ve:
                print(f"Value error: {ve}")
                break  # Exit loop if no data received

            #sleep(5)  # Sleep for a bit before the next iteration

    except (socket.error, ValueError) as e:
        print(f"Error: {e}")

    finally:
        client_socket.close()

if __name__ == "__main__":
    main()
''')
print("Alice's script saved to alice_client.py")

Alice's script saved to alice_client.py


In [6]:
### Starting the Simulation
'''
**Important**: Start the nodes in the following order:

1. **Bob** (Server - listens for connections)

2. **Eve** (Relay - connects to Bob and waits for Alice)

3. **Alice** (Client - connects to Eve)

'''

'\n**Important**: Start the nodes in the following order:\n\n1. **Bob** (Server - listens for connections)\n   \n2. **Eve** (Relay - connects to Bob and waits for Alice)\n   \n3. **Alice** (Client - connects to Eve)\n   \n'

In [7]:
!nohup python bob_server.py > bob_output.log 2>&1 &
!sleep 5
!tail    bob_output.log

In [8]:
!nohup python eve_server.py > eve_output.log 2>&1 &
!sleep 5
!tail    eve_output.log

In [9]:
!nohup python alice_client.py > alice_output.log 2>&1 &
!sleep 5
!tail    alice_output.log

In [10]:
!tail -f eve_output.log

Eve is listening for connections...
Connected to Alice from ('127.0.0.1', 53070)
Connected to Bob.
Eve received Alice's data: [1, 1, 1, 1, 0, 0, 0, 1]|[0, 1, 0, 0, 1, 0, 1, 1]
Data sent to Bob: [1, 1, 1, 1, 0, 0, 0, 1]|[0, 1, 0, 0, 1, 0, 1, 1]
Eve received Bob's response: [1, 0, 0, 0, 1, 1, 1, 1]
Response sent to Alice: [1, 0, 0, 0, 1, 1, 1, 1]
Iteration: 1, Eve's success rate: 0.00000%
Eve received Alice's data: [1, 0, 1, 0, 0, 0, 1, 0]|[0, 0, 0, 0, 0, 1, 0, 1]
Data sent to Bob: [1, 0, 1, 0, 0, 0, 1, 0]|[0, 0, 0, 0, 0, 1, 0, 1]
Eve received Bob's response: [0, 1, 0, 0, 0, 1, 0, 1]
Response sent to Alice: [0, 1, 0, 0, 0, 1, 0, 1]
Iteration: 2, Eve's success rate: 0.00000%
Eve received Alice's data: [0, 0, 1, 1, 1, 1, 1, 1]|[0, 1, 0, 1, 0, 0, 1, 1]
Data sent to Bob: [0, 0, 1, 1, 1, 1, 1, 1]|[0, 1, 0, 1, 0, 0, 1, 1]
Eve received Bob's response: [1, 1, 1, 1, 1, 1, 0, 0]
Response sent to Alice: [1, 1, 1, 1, 1, 1, 0, 0]
Iteration: 3, Eve's success rate: 0.00000%
Eve received Alice's data: [